Recommended

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # MUST be set before numpy/pandas import; big ALS speedup on Colab

Connect To Drive and import Libraries

In [3]:

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

PROJECT_ROOT = '/content/drive/MyDrive/Projects/multi-agent-discovery'
DATA = os.path.join(PROJECT_ROOT, 'data/raw/ml-32m')
ART  = os.path.join(PROJECT_ROOT, 'artifacts/als')
os.makedirs(ART, exist_ok=True)
print("root exists:", os.path.exists(PROJECT_ROOT))

Mounted at /content/drive
root exists: True


install and import implicit

In [4]:
!pip install implicit -q
from implicit.als import AlternatingLeastSquares
import implicit
print("implicit version:", implicit.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 78.8 MB/s eta 0:00:00
implicit version: 0.7.3


load ratings and apply the "liked" threshold

In [5]:
ratings = pd.read_csv(
    os.path.join(DATA, 'ratings.csv'),
    usecols=['userId', 'movieId', 'rating'],
    dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'},
)

LIKE_THRESHOLD = 3.5                                            # <------ what rating qualies a movie to be recommended
liked = ratings[ratings['rating'] >= LIKE_THRESHOLD].copy()
print(f"kept {len(liked):,} of {len(ratings):,} ratings ({len(liked)/len(ratings)*100:.1f}%)")

kept 20,228,336 of 32,000,204 ratings (63.2%)


build contiguous ID mappings

In [6]:
# MovieLens IDs are non-contiguous with gaps; ALS needs dense 0..N-1 indices for rows and cols.
liked['user_cat'] = liked['userId'].astype('category')
liked['item_cat'] = liked['movieId'].astype('category')

user_categories = liked['user_cat'].cat.categories.to_numpy()   # position i -> original userId
item_categories = liked['item_cat'].cat.categories.to_numpy()   # position j -> original movieId

user_codes = liked['user_cat'].cat.codes.to_numpy()             # 0..n_users-1
item_codes = liked['item_cat'].cat.codes.to_numpy()             # 0..n_items-1
confidence = liked['rating'].to_numpy(dtype='float32')          # raw rating as confidence value

n_users, n_items = len(user_categories), len(item_categories)
print(f"n_users={n_users:,}  n_items={n_items:,}")

n_users=200,808  n_items=65,032


assemble the sparse user×item matrix

In [7]:
# CRITICAL ORIENTATION: modern implicit wants (users x items) for fit().
user_item = csr_matrix((confidence, (user_codes, item_codes)), shape=(n_users, n_items))
print("matrix:", user_item.shape, "| stored interactions:", user_item.nnz, "| dtype:", user_item.dtype)

matrix: (200808, 65032) | stored interactions: 20228336 | dtype: float32


train ALS

In [8]:
model = AlternatingLeastSquares(
    factors=64,            # width of each user/movie vector
    regularization=0.05,   # ridge penalty; guards against overfitting
    iterations=15,         # alternating sweeps
    random_state=42,
)
model.fit(user_item)       # <- pass the (users x items) matrix directly

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

sanity check

In [9]:
def titles_for(movie_ids):
    m = pd.read_csv(os.path.join(DATA, 'movies.csv'))
    return m[m['movieId'].isin(movie_ids)][['movieId', 'title', 'genres']]

demo_user_id = 1
u_idx = int(np.where(user_categories == demo_user_id)[0][0])   # userId -> row index

# What did this user actually like? (their signal)
liked_by_user = liked[liked['userId'] == demo_user_id].nlargest(8, 'rating')['movieId'].tolist()
print("USER LIKED:\n", titles_for(liked_by_user)[['title']].to_string(index=False), "\n")

# What does ALS recommend?
ids, scores = model.recommend(u_idx, user_item[u_idx], N=10, filter_already_liked_items=True)
rec_movie_ids = item_categories[ids]                           # item indices -> movieIds
print("ALS RECOMMENDS:\n", titles_for(rec_movie_ids)[['title', 'genres']].to_string(index=False))

USER LIKED:
                                                title
Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
           Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
         White Balloon, The (Badkonake sefid) (1995)
                                  Taxi Driver (1976)
                         Doom Generation, The (1995)
         Eat Drink Man Woman (Yin shi nan nu) (1994)
           Star Wars: Episode IV - A New Hope (1977)
    Three Colors: Red (Trois couleurs: Rouge) (1994) 

ALS RECOMMENDS:
                                                                       title                           genres
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)                       Comedy|War
                                                       Trainspotting (1996)               Comedy|Crime|Drama
                                                          Casablanca (1942)                    Drama|Romance
                                               2001: A 

save the model + mappings

In [10]:
model.save(os.path.join(ART, 'als_model.npz'))
np.savez(os.path.join(ART, 'als_mappings.npz'),
         user_categories=user_categories, item_categories=item_categories)
print("saved to:", ART, "->", os.listdir(ART))

saved to: /content/drive/MyDrive/Projects/multi-agent-discovery/artifacts/als -> ['als_model.npz', 'als_mappings.npz']


load the trained model, rebuild + save the interaction matrix

In [13]:

from implicit.cpu.als import AlternatingLeastSquares as ALS
from scipy.sparse import csr_matrix, save_npz, load_npz

# Load the trained model + id mappings — NO retraining
model = ALS.load(os.path.join(ART, 'als_model.npz'))
maps = np.load(os.path.join(ART, 'als_mappings.npz'))
user_categories, item_categories = maps['user_categories'], maps['item_categories']
n_items = len(item_categories)
print("loaded model + mappings | n_users:", len(user_categories), "| n_items:", n_items)

# Rebuild the (users x items) matrix using the SAVED mappings (keeps factor indices aligned).
ratings = pd.read_csv(os.path.join(DATA, 'ratings.csv'),
                      usecols=['userId','movieId','rating'],
                      dtype={'userId':'int32','movieId':'int32','rating':'float32'})
liked = ratings[ratings['rating'] >= 3.5]
u_map = {int(u): i for i, u in enumerate(user_categories)}
i_map = {int(m): j for j, m in enumerate(item_categories)}
liked = liked[liked['userId'].isin(u_map) & liked['movieId'].isin(i_map)]
user_item = csr_matrix((liked['rating'].to_numpy('float32'),
                        (liked['userId'].map(u_map).to_numpy(),
                         liked['movieId'].map(i_map).to_numpy())),
                       shape=(len(user_categories), n_items))
save_npz(os.path.join(ART, 'user_item.npz'), user_item)
print("rebuilt + saved user_item:", user_item.shape, "| nnz:", user_item.nnz)

loaded model + mappings | n_users: 200808 | n_items: 65032
rebuilt + saved user_item: (200808, 65032) | nnz: 20228336


lookup helpers

In [14]:
movies    = pd.read_csv(os.path.join(DATA, 'movies.csv'))[['movieId','title','genres']]
movies_ix = movies.set_index('movieId')

def find_movie(substr):     # find a movieId by title text
    return movies[movies['title'].str.contains(substr, case=False, na=False)][['movieId','title']].head(10)

def titles(movie_ids):      # movieIds -> title/genre rows, order preserved, missing dropped
    present = [m for m in movie_ids if m in movies_ix.index]
    return movies_ix.loc[present].reset_index()

print(find_movie('Lion King').to_string(index=False))   # confirm the anchor IDs you'll use

 movieId                 title
     364 Lion King, The (1994)
  203222  The Lion King (2019)


Mode 1 (demo-user) and test

In [15]:
def candidates_for_user(user_id, n=200):
    if user_id not in u_map:
        return []                                            # unknown user
    u = u_map[user_id]
    ids, _ = model.recommend(u, user_item[u], N=n, filter_already_liked_items=True)
    return [int(item_categories[i]) for i in ids]

print(titles(candidates_for_user(1, n=10))[['title','genres']].to_string(index=False))

                                                                      title                           genres
                                                          Casablanca (1942)                    Drama|Romance
                                               2001: A Space Odyssey (1968)           Adventure|Drama|Sci-Fi
                                                     American Beauty (1999)                    Drama|Romance
                                                   L.A. Confidential (1997) Crime|Film-Noir|Mystery|Thriller
                                                           Chinatown (1974) Crime|Film-Noir|Mystery|Thriller
                                                               Alien (1979)                    Horror|Sci-Fi
                          Star Wars: Episode VI - Return of the Jedi (1983)          Action|Adventure|Sci-Fi
                                                  Lawrence of Arabia (1962)              Adventure|Drama|War
Dr. Strangelove or:

Mode 2 (anchor fold-in) and test

In [16]:
def candidates_from_anchors(anchor_movie_ids, n=200):
    idxs    = [i_map[m] for m in anchor_movie_ids if m in i_map]
    dropped = [m for m in anchor_movie_ids if m not in i_map]   # anchors outside ALS space
    if not idxs:
        return [], dropped
    row = csr_matrix((np.ones(len(idxs), 'float32'),
                      (np.zeros(len(idxs), 'int32'), np.array(idxs, 'int32'))),
                     shape=(1, n_items))                         # ONE-row pseudo-user
    ids, _ = model.recommend(0, row, N=n,
                             filter_already_liked_items=True,    # auto-excludes the anchors
                             recalculate_user=True)              # compute taste vector from anchors
    return [int(item_categories[i]) for i in ids], dropped

anchors = [1, 364]   # 1 = Toy Story, 364 = The Lion King  (verify via find_movie above)
recs, dropped = candidates_from_anchors(anchors, n=10)
print("anchors:", titles(anchors)['title'].tolist(), "| dropped:", dropped, "\n")
print(titles(recs)[['title','genres']].to_string(index=False))

anchors: ['Toy Story (1995)', 'Lion King, The (1994)'] | dropped: [] 

                      title                                              genres
             Aladdin (1992)         Adventure|Animation|Children|Comedy|Musical
         Toy Story 2 (1999)         Adventure|Animation|Children|Comedy|Fantasy
Beauty and the Beast (1991)     Animation|Children|Fantasy|Musical|Romance|IMAX
      Monsters, Inc. (2001)         Adventure|Animation|Children|Comedy|Fantasy
        Finding Nemo (2003)                 Adventure|Animation|Children|Comedy
         Toy Story 3 (2010)    Adventure|Animation|Children|Comedy|Fantasy|IMAX
                Babe (1995)                                      Children|Drama
       Bug's Life, A (1998)                 Adventure|Animation|Children|Comedy
               Shrek (2001) Adventure|Animation|Children|Comedy|Fantasy|Romance
        Forrest Gump (1994)                            Comedy|Drama|Romance|War


utility (similar_movies) and test

In [17]:
def similar_movies(movie_id, n=20):
    if movie_id not in i_map:
        return []
    ids, _ = model.similar_items(i_map[movie_id], N=n+1)         # +1: results include the movie itself
    return [int(item_categories[j]) for j in ids if int(item_categories[j]) != movie_id][:n]

print(titles(similar_movies(1, 8))[['title','genres']].to_string(index=False))   # near Toy Story

                                     title                                           genres
                        Toy Story 2 (1999)      Adventure|Animation|Children|Comedy|Fantasy
                     Monsters, Inc. (2001)      Adventure|Animation|Children|Comedy|Fantasy
                     Lion King, The (1994)  Adventure|Animation|Children|Drama|Musical|IMAX
                            Aladdin (1992)      Adventure|Animation|Children|Comedy|Musical
                       Finding Nemo (2003)              Adventure|Animation|Children|Comedy
                      Bug's Life, A (1998)              Adventure|Animation|Children|Comedy
                        Toy Story 3 (2010) Adventure|Animation|Children|Comedy|Fantasy|IMAX
Willy Wonka & the Chocolate Factory (1971)                  Children|Comedy|Fantasy|Musical


package the tool as an importable module

In [20]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_a_als.py
"""Tool A — ALS collaborative-filtering candidate generator."""
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz
from implicit.cpu.als import AlternatingLeastSquares


class ToolA:
    def __init__(self, artifacts_dir, movies_csv, user_item_path=None):
        self.model = AlternatingLeastSquares.load(os.path.join(artifacts_dir, "als_model.npz"))
        maps = np.load(os.path.join(artifacts_dir, "als_mappings.npz"))
        self.item_categories = maps["item_categories"]
        self.user_categories = maps["user_categories"]
        self.n_items = len(self.item_categories)
        self.i_map = {int(m): j for j, m in enumerate(self.item_categories)}
        self.u_map = {int(u): i for i, u in enumerate(self.user_categories)}
        self._movies = pd.read_csv(movies_csv)[["movieId", "title", "genres"]].set_index("movieId")
        self.user_item = load_npz(user_item_path) if user_item_path else None

    def titles(self, movie_ids):
        present = [m for m in movie_ids if m in self._movies.index]
        return self._movies.loc[present].reset_index().to_dict("records")

    def candidates_from_anchors(self, anchor_movie_ids, n=200):
        idxs = [self.i_map[m] for m in anchor_movie_ids if m in self.i_map]
        dropped = [m for m in anchor_movie_ids if m not in self.i_map]
        if not idxs:
            return {"candidates": [], "dropped_anchors": dropped}
        row = csr_matrix((np.ones(len(idxs), "float32"),
                          (np.zeros(len(idxs), "int32"), np.array(idxs, "int32"))),
                         shape=(1, self.n_items))
        ids, _ = self.model.recommend(0, row, N=n,
                                      filter_already_liked_items=True, recalculate_user=True)
        return {"candidates": [int(self.item_categories[i]) for i in ids],
                "dropped_anchors": dropped}

    def candidates_for_user(self, user_id, n=200):
        if self.user_item is None:
            raise RuntimeError("demo-user mode needs user_item_path")
        if user_id not in self.u_map:
            return {"candidates": [], "error": "unknown user_id"}
        u = self.u_map[user_id]
        ids, _ = self.model.recommend(u, self.user_item[u], N=n, filter_already_liked_items=True)
        return {"candidates": [int(self.item_categories[i]) for i in ids]}

    def similar_movies(self, movie_id, n=20):
        if movie_id not in self.i_map:
            return {"similar": [], "error": "movie not in ALS space"}
        ids, _ = self.model.similar_items(self.i_map[movie_id], N=n + 1)
        out = [int(self.item_categories[j]) for j in ids if int(self.item_categories[j]) != movie_id]
        return {"similar": out[:n]}

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/tools/tool_a_als.py


import the module fresh and smoke-test

In [21]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
sys.modules.pop('tools.tool_a_als', None)
from tools.tool_a_als import ToolA

tool_a = ToolA(artifacts_dir=ART,
               movies_csv=os.path.join(DATA, 'movies.csv'),
               user_item_path=os.path.join(ART, 'user_item.npz'))

print("anchors ->", [r['title'] for r in tool_a.titles(tool_a.candidates_from_anchors([1,364], n=5)['candidates'])])
print("user 1  ->", [r['title'] for r in tool_a.titles(tool_a.candidates_for_user(1, n=5)['candidates'])])
print("similar ->", [r['title'] for r in tool_a.titles(tool_a.similar_movies(1, n=5)['similar'])])

anchors -> ['Aladdin (1992)', 'Toy Story 2 (1999)', 'Beauty and the Beast (1991)', 'Monsters, Inc. (2001)', 'Finding Nemo (2003)']
user 1  -> ['Casablanca (1942)', '2001: A Space Odyssey (1968)', 'American Beauty (1999)', 'L.A. Confidential (1997)', 'Chinatown (1974)']
similar -> ['Toy Story 2 (1999)', 'Monsters, Inc. (2001)', 'Lion King, The (1994)', 'Aladdin (1992)', 'Finding Nemo (2003)']
